# Code-GraphRAG System - Complete End-to-End Notebook
This notebook demonstrates all 5 phases of the **Code-GraphRAG Architecture**:
1. **Infrastructure Verification**: Testing **Neo4j**, **ChromaDB**, and **Ollama (`gemma4:31b-cloud`)** connections.
2. **Dual-Brain Indexing**: Synchronizing vector embeddings (ChromaDB) and AST structural knowledge graphs (Neo4j).
3. **Surgical Purge & Webhook Sync**: Simulating Git commit file updates and dynamic re-indexing.
4. **Vector Search & Graph Blast Radius Traversal**: Querying line locations, call graphs, and affected caller components.
5. **Code Rewrite Generation**: Using `gemma4:31b-cloud` to produce exact line-targeted code rewrites and impact warnings.

In [ ]:
import os
import json
from dual_index_sync import DualIndexSync
from graph_rag_pipeline import CodeGraphRAGPipeline
from webhook_server import app

print("✓ All Code-GraphRAG modules loaded successfully!")

## Step 1: Initialize Dual-Brain Storage (Neo4j + ChromaDB)
Perform full repository synchronization: AST parsing, vector chunking, OKF triple extraction, and Neo4j graph storage.

In [ ]:
sync = DualIndexSync()
sync_summary = sync.sync_repository(repo_target=".")
print("\n=== Dual-Brain Synchronization Summary ===")
print(json.dumps(sync_summary, indent=2))
sync.close()

## Step 2: Test Surgical Purge & Dynamic Re-indexing
Simulate a modified file update (purging stale embeddings and graph nodes, then upserting fresh representations).

In [ ]:
sync = DualIndexSync()
target_file = os.path.abspath("dual_index_sync.py")
print(f"Simulating dynamic webhook sync for: {target_file}")

res = sync.sync_file(file_path=target_file, repo_root=os.path.abspath("."), repo_name="Code-Helper")
print("Surgical update result:", res)
sync.close()

## Step 3: Run Vector Semantic Search & Graph Blast Radius Traversal
Retrieve semantic code blocks from ChromaDB and trace dependency call chains in Neo4j.

In [ ]:
pipeline = CodeGraphRAGPipeline()

# 1. Semantic Vector Search
query = "sync_file function for purging stale nodes"
vector_chunks = pipeline.search_vector(query, n_results=2)
print("=== ChromaDB Vector Results ===")
for vc in vector_chunks:
    print(f"File: {vc['metadata'].get('file_path')} (Lines {vc['metadata'].get('line_start')}-{vc['metadata'].get('line_end')})")
    print(vc["code"][:200] + "...\n")

# 2. Graph Traversal & Blast Radius Analysis
blast = pipeline.traverse_graph_blast_radius("sync_file")
print("=== Neo4j Graph Blast Radius Context ===")
print(json.dumps(blast, indent=2))
pipeline.close()

## Step 4: Execute Full Code-GraphRAG Query & Architect Code Rewrite
Prompt the Code-GraphRAG engine to generate target file locations, exact line numbers, code rewrites, and impact warnings.

In [ ]:
pipeline = CodeGraphRAGPipeline()
user_prompt = "Update the sync_file function to handle error logging when ChromaDB is unreachable."

rag_result = pipeline.query_code_graph_rag(user_prompt)
print("\n==========================================")
print("GEMMA 4 30B CODE REWRITE & BLAST WARNING:")
print("==========================================")
print(rag_result["answer"])
pipeline.close()